# 09 · Evaluación de recuperación léxica vs. RDF (compatible con Jupyter)

**Sin `ipywidgets`, Colab, OCR ni descargas nuevas.** Este notebook usa los resultados de `etapa7.zip` y `etapa8.zip`, y conserva las evaluaciones ya guardadas.

1. Ejecuta **Run All Cells**. Se genera `data/output/etapa9.zip`.
2. Para evaluar una pregunta, en una celda nueva escribe `evaluar_siguiente()` y ejecuta. Repite para las 12 preguntas.
3. Se guardan los juicios después de cada pregunta y se actualiza automáticamente `etapa9.zip`.

**Escala:** `0` = no relevante; `1` = relación indirecta; `2` = directamente relevante. Si no conoces la respuesta, deja la pregunta sin calificar. La evaluación se realiza sobre las fichas catalográficas, no sobre la lectura completa de cada obra.


In [1]:
from pathlib import Path
from IPython.display import display
import io, json, math, re, unicodedata, zipfile, hashlib
from datetime import datetime, timezone
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cwd = Path.cwd().resolve()
if (cwd/'notebooks').is_dir() and (cwd/'data').is_dir(): ROOT=cwd
elif cwd.name=='notebooks' and (cwd.parent/'data').is_dir(): ROOT=cwd.parent
elif (cwd.parent/'data').is_dir() and (cwd.parent/'notebooks').is_dir(): ROOT=cwd.parent
else: raise FileNotFoundError('Abre el notebook dentro de notebooks/ del proyecto memoria_chilena.')

def localizar(etapa):
    for p in [ROOT/'data'/'output'/f'etapa{etapa}.zip',ROOT/'data'/'input'/f'etapa{etapa}.zip']:
        if p.exists():return p
    raise FileNotFoundError(f'Falta etapa{etapa}.zip. Ejecuta la etapa correspondiente.')

Z7=localizar(7);Z8=localizar(8)
SALIDA=ROOT/'data'/'output'/'etapa9';SALIDA.mkdir(parents=True,exist_ok=True)
print('Proyecto:',ROOT)
print('Entradas:',Z7.name, Z8.name)
print('Salidas:',SALIDA)

Proyecto: C:\Users\jvera\Documents\memoria_chilena
Entradas: etapa7.zip etapa8.zip
Salidas: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa9


In [2]:
with zipfile.ZipFile(Z7) as z:
    catalogo=pd.read_csv(io.BytesIO(z.read('catalogo_para_busqueda.csv')),dtype=str).fillna('')
with zipfile.ZipFile(Z8) as z:
    relaciones=pd.read_csv(io.BytesIO(z.read('relaciones_filtradas.csv')),dtype=str).fillna('')
    frecs=pd.read_csv(io.BytesIO(z.read('unidades_frecuencias.csv')),dtype=str).fillna('')
    evaluacion8=pd.read_csv(io.BytesIO(z.read('PLANTILLA_EVALUACION_MANUAL.csv')),dtype=str).fillna('')
    informe8=json.loads(z.read('informe_etapa8.json').decode('utf-8'))

assert len(catalogo)==10 and catalogo['mcid'].is_unique, 'Se esperaban 10 fichas sin MCID duplicado'
assert len(relaciones)==14,'La etapa 8 no tiene las 14 conexiones esperadas'
consultas=list(dict.fromkeys(evaluacion8['pregunta'].tolist()))
assert len(consultas)==12, 'Se esperaban 12 preguntas'
assert set(relaciones['mcid_origen']).union(relaciones['mcid_destino']).issubset(set(catalogo['mcid']))
print(f'Fichas: {len(catalogo)} | Relaciones filtradas: {len(relaciones)} | Consultas: {len(consultas)}')

Fichas: 10 | Relaciones filtradas: 14 | Consultas: 12


## 1. Dos métodos comparables

Se reproduce el método léxico de la etapa 8 y se agrega una variante que difunde el puntaje a través de relaciones curatoriales específicas. **No se ajustan pesos según los juicios humanos**; fijamos 80/20 antes de obtenerlos. Las etiquetas universales permanecen en los datos originales, pero no contribuyen a la expansión.

In [3]:
def normalizar(s):
    s=unicodedata.normalize('NFKD',str(s).lower())
    s=''.join(c for c in s if not unicodedata.combining(c))
    s=re.sub(r'[^a-z0-9\s]',' ',s)
    return re.sub(r'\s+',' ',s).strip()

ALIASES={'quichua':'quechua','piru':'peru','yncas':'incas'}
def expandir_alias(s):
    s=normalizar(s)
    for a,b in ALIASES.items():s=re.sub(r'\b'+a+r'\b',b,s)
    return s

def separar_unidades(texto):
    return {x.strip() for x in str(texto).split('|') if x.strip()}

universales=set(frecs.loc[frecs['uso_expansion'].str.contains('excluir',case=False),'unidad'])
por_doc={r['mcid']:separar_unidades(r['unidades']) for _,r in catalogo.iterrows()}

def texto_registro(r):
    partes=[]
    for col,repeticiones in [('titulo',4),('autor',2),('materias',1),('notas',2),('datos',1)]:
        partes.extend([expandir_alias(r[col])]*repeticiones)
    partes.extend([expandir_alias(' '.join(sorted(por_doc[r['mcid']]-universales)))]*2)
    return ' '.join(partes)

corpus=[texto_registro(r) for _,r in catalogo.iterrows()]
vec_p=TfidfVectorizer(ngram_range=(1,2),sublinear_tf=True,token_pattern=r'(?u)\b\w\w+\b')
vec_c=TfidfVectorizer(analyzer='char',ngram_range=(3,5),sublinear_tf=True)
Xp=vec_p.fit_transform(corpus); Xc=vec_c.fit_transform(corpus)
ids=catalogo['mcid'].tolist(); posiciones={x:i for i,x in enumerate(ids)}

# Matriz de vecindad normalizada por fila. Los nodos aislados no reciben propagación.
A=np.zeros((len(ids),len(ids)),float)
for _,r in relaciones.iterrows():
    i=posiciones[r['mcid_origen']];j=posiciones[r['mcid_destino']]
    w=float(r['peso_idf'])
    A[i,j]=A[j,i]=w
filas=A.sum(axis=1)
P=np.divide(A,filas[:,None],out=np.zeros_like(A),where=(filas[:,None]!=0))

PESO_LEXICO=0.80
PESO_GRAFO=0.20

def calcular_puntajes(pregunta):
    q=expandir_alias(pregunta)
    p=cosine_similarity(vec_p.transform([q]),Xp)[0]
    c=cosine_similarity(vec_c.transform([q]),Xc)[0]
    lex=0.7*p+0.3*c
    # Promedio ponderado de puntajes léxicos de los vecinos temáticos, NO nuevos hechos.
    graph=P @ lex
    mixto=PESO_LEXICO*lex+PESO_GRAFO*graph
    return lex,mixto,graph

ranking=[]
for pregunta in consultas:
    lex,mixto,vec=calcular_puntajes(pregunta)
    for metodo,scores in [('lexico',lex),('lexico_rdf',mixto)]:
        orden=sorted(range(len(ids)),key=lambda i:(-float(scores[i]),ids[i]))
        for rank,i in enumerate(orden,1):
            ranking.append({'pregunta':pregunta,'metodo':metodo,'posicion':rank,
                           'mcid':ids[i],'titulo':catalogo.iloc[i]['titulo'],
                           'ficha':catalogo.iloc[i]['ficha'],
                           'score':round(float(scores[i]),7),
                           'score_lexico':round(float(lex[i]),7),
                           'componente_vecindad':round(float(vec[i]),7)})
rank_df=pd.DataFrame(ranking)
assert len(rank_df)==12*2*10
rank_df.to_csv(SALIDA/'rankings_comparados.csv',index=False,encoding='utf-8-sig')
print('Se generaron',len(rank_df),'posiciones (12 consultas × 10 documentos × 2 métodos).')

Se generaron 240 posiciones (12 consultas × 10 documentos × 2 métodos).


In [4]:
# Diagnóstico descriptivo: no se presenta como medición de relevancia.
resumen_top=[]
for pregunta in consultas:
    a=rank_df[(rank_df.pregunta==pregunta)&(rank_df.metodo=='lexico')].iloc[0]
    b=rank_df[(rank_df.pregunta==pregunta)&(rank_df.metodo=='lexico_rdf')].iloc[0]
    resumen_top.append({'pregunta':pregunta,'top1_lexico':a.mcid,
                        'top1_lexico_rdf':b.mcid,'cambio_top1':a.mcid!=b.mcid,
                        'titulo_lexico':a.titulo,'titulo_lexico_rdf':b.titulo})
top_df=pd.DataFrame(resumen_top)
top_df.to_csv(SALIDA/'cambios_top1.csv',index=False,encoding='utf-8-sig')
print('Preguntas donde cambió el primer resultado:',int(top_df.cambio_top1.sum()),'de',len(top_df))
# El mismo top-1 no implica rankings idénticos en los puestos inferiores.
cambios_orden=0;cambios_top5=0
for pregunta in consultas:
    a=rank_df[(rank_df.pregunta==pregunta)&(rank_df.metodo=='lexico')].sort_values('posicion').mcid.tolist()
    b=rank_df[(rank_df.pregunta==pregunta)&(rank_df.metodo=='lexico_rdf')].sort_values('posicion').mcid.tolist()
    cambios_orden+=int(a!=b)
    cambios_top5+=int(set(a[:5])!=set(b[:5]))
resumen_movimientos={'top1_cambio_consultas':int(top_df.cambio_top1.sum()),
                     'orden_cambio_consultas':cambios_orden,
                     'membresia_top5_cambio_consultas':cambios_top5,
                     'total_consultas':len(consultas),
                     'no_demuestra_mejora':True}
(SALIDA/'diagnostico_movimientos.json').write_text(json.dumps(resumen_movimientos,ensure_ascii=False,indent=2),encoding='utf-8')
print('Cambios de orden en',cambios_orden,'consultas; cambios de integrantes del top-5 en',cambios_top5)
display(top_df[['pregunta','top1_lexico','top1_lexico_rdf','cambio_top1']])

Preguntas donde cambió el primer resultado: 0 de 12
Cambios de orden en 8 consultas; cambios de integrantes del top-5 en 5


,pregunta,top1_lexico,top1_lexico_rdf,cambio_top1
0,¿Qué obras catalogadas tratan sobre el aymara?,MC0014755,MC0014755,False
1,¿Qué textos coloniales están vinculados a la e...,MC0038283,MC0038283,False
2,¿Qué documentos relacionan misiones jesuitas y...,MC0014753,MC0014753,False
3,¿Qué obras corresponden a cronistas del Perú?,MC0014761,MC0014761,False
4,¿Qué documentos permiten estudiar la geografía...,MC0038283,MC0038283,False
5,¿Qué fuentes se relacionan con la fiesta de La...,MC0023126,MC0023126,False
6,¿Qué obras están catalogadas sobre bailes chinos?,MC0038283,MC0038283,False
7,¿Qué materiales hay acerca de idolatría en el ...,MC0038290,MC0038290,False
8,¿Qué obras se relacionan con el Tawantinsuyu?,MC0038290,MC0038290,False
9,¿Qué fuentes están vinculadas con Historia nat...,MC0014757,MC0014757,False


## 2. Juicios humanos sobre las 120 combinaciones

La etapa 8 proponía solo **5 documentos por pregunta**. Para evaluar realmente Recall@5 y nDCG@5 necesitamos saber si hay documentos pertinentes **fuera** de los cinco primeros. Por eso usamos los **10 documentos del corpus × 12 preguntas = 120 juicios**.

Se prepara una tabla **ciega al ranking**, con valores:

- **0:** no relevante para la pregunta, según los metadatos disponibles.
- **1:** relación parcial o indirecta con la pregunta.
- **2:** documento directamente pertinente, verificable en los metadatos de la ficha.

Si los metadatos no permiten decidir, deja la celda vacía. **No adivines el contenido de libros a partir de su título.** Los juicios se conservan al volver a ejecutar el notebook.

In [5]:
JUICIOS=SALIDA/'juicios_relevancia.csv'

def orden_ciego(q):
    return sorted(ids,key=lambda i:hashlib.sha256((q+'|'+i+'|mc_eval_v1').encode('utf-8')).hexdigest())

filas=[]
for pregunta in consultas:
    for mcid in orden_ciego(pregunta):
        r=catalogo.loc[catalogo.mcid==mcid].iloc[0]
        filas.append({'pregunta':pregunta,'mcid':mcid,'titulo':r['titulo'],
                      'autor':r['autor'],'materias':r['materias'],'unidades':r['unidades'],
                      'ficha':r['ficha'],'relevancia_0_1_2':'','justificacion':'','evaluador':''})
qrels=pd.DataFrame(filas)
if JUICIOS.exists():
    anteriores=pd.read_csv(JUICIOS,dtype=str).fillna('')
    anteriores=anteriores.drop_duplicates(['pregunta','mcid'],keep='last')
    col_keep=['pregunta','mcid','relevancia_0_1_2','justificacion','evaluador']
    qrels=qrels.drop(columns=['relevancia_0_1_2','justificacion','evaluador']).merge(
        anteriores[col_keep],on=['pregunta','mcid'],how='left').fillna('')
qrels.to_csv(JUICIOS,index=False,encoding='utf-8-sig')
print('Juicios rellenados:',int(qrels.relevancia_0_1_2.isin(['0','1','2']).sum()),'/120')
print('Tabla:',JUICIOS)

Juicios rellenados: 0 /120
Tabla: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa9\juicios_relevancia.csv


## 3. Métricas: se calculan **solo con todos los juicios completos**

Con las 120 etiquetas válidas calculamos **nDCG@5**, **Precision@5**, **Recall@5** y **MRR@10** de ambos métodos. Si faltan juicios, se guarda el diagnóstico y se indica el estado *pendiente*; no se confunde un valor vacío con relevancia cero.

In [6]:
def actualizar_evaluacion():
    juicios=pd.read_csv(JUICIOS,dtype=str).fillna('')
    labels={(r.pregunta,r.mcid):r.relevancia_0_1_2 for r in juicios.itertuples(index=False)}
    pendientes=sum(x not in ['0','1','2'] for x in labels.values())
    assert len(labels)==120, 'La tabla debe tener las 120 combinaciones pregunta–documento'
    state={'etapa':9,'fuente':['etapa7.zip','etapa8.zip'],'generado_utc':datetime.now(timezone.utc).isoformat(),
           'n_documentos':len(ids),'n_consultas':len(consultas),'n_relaciones_especificas':len(relaciones),
           'metodos':['lexico','lexico_rdf'],'pesos':{'lexico':PESO_LEXICO,'grafo':PESO_GRAFO},
           'etiquetas_validas':120-pendientes,'etiquetas_pendientes':pendientes,
           'estado':'evaluacion_completa' if pendientes==0 else 'pendiente_de_juicios_humanos',
           'advertencias':['co-membresía RDF no es relación histórica','las etiquetas de relevancia se basan en metadatos, no en lectura íntegra','no interpretar puntuaciones de similitud como probabilidades']}
    if pendientes==0:
        metricas=[]
        for pregunta in consultas:
            gold={mcid:int(labels[(pregunta,mcid)]) for mcid in ids}
            relevant=sum(v>=1 for v in gold.values())
            ideal=sorted(gold.values(),reverse=True)
            def dcg(vals,k=5):return sum((2**v-1)/math.log2(rank+2) for rank,v in enumerate(vals[:k]))
            idcg=dcg(ideal)
            for metodo in ['lexico','lexico_rdf']:
                orden=rank_df[(rank_df.pregunta==pregunta)&(rank_df.metodo==metodo)].sort_values('posicion').mcid.tolist()
                vals=[gold[i] for i in orden]
                hit5=sum(v>=1 for v in vals[:5])
                first=next((i+1 for i,v in enumerate(vals) if v>=1),None)
                metricas.append({'pregunta':pregunta,'metodo':metodo,
                                 'precision_at_5':hit5/5,
                                 'recall_at_5':hit5/relevant if relevant else float('nan'),
                                 'ndcg_at_5':dcg(vals)/idcg if idcg else float('nan'),
                                 'mrr_at_10':1/first if first else 0,
                                 'n_relevantes':relevant})
        m=pd.DataFrame(metricas)
        m.to_csv(SALIDA/'metricas_por_consulta.csv',index=False,encoding='utf-8-sig')
        means=m.groupby('metodo')[['precision_at_5','recall_at_5','ndcg_at_5','mrr_at_10']].mean().reset_index()
        means.to_csv(SALIDA/'metricas_resumen.csv',index=False,encoding='utf-8-sig')
        state['metricas_promedio']={r['metodo']:{k:float(r[k]) for k in ['precision_at_5','recall_at_5','ndcg_at_5','mrr_at_10']} for _,r in means.iterrows()}
    else:
        for filename in ['metricas_por_consulta.csv','metricas_resumen.csv']:
            if (SALIDA/filename).exists():(SALIDA/filename).unlink()
    (SALIDA/'informe_etapa9.json').write_text(json.dumps(state,ensure_ascii=False,indent=2),encoding='utf-8')
    return state


def crear_zip():
    destino=ROOT/'data'/'output'/'etapa9.zip'
    with zipfile.ZipFile(destino,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(SALIDA.iterdir()):
            if p.is_file():z.write(p,arcname=p.name)
    return destino

state=actualizar_evaluacion()
final_zip=crear_zip()
print('Estado:',state['estado'],'| Etiquetas:',state['etiquetas_validas'],'/120')
print('ZIP automático:',final_zip)

Estado: pendiente_de_juicios_humanos | Etiquetas: 0 /120
ZIP automático: C:\Users\jvera\Documents\memoria_chilena\data\output\etapa9.zip


## 4. Evaluación humana **sin widgets**

En vez de controles interactivos que pueden fallar en Jupyter, utiliza una celda nueva con:

```python
evaluar_siguiente()
```

El notebook mostrará la pregunta y diez fichas (ordenadas de forma independiente del buscador), con sus enlaces. Escribe **diez notas**, separadas por espacios, por ejemplo `2 0 1 0 0 2 1 0 0 1`, y pulsa Intro.

La respuesta se guardará inmediatamente en `data/output/etapa9/juicios_relevancia.csv` y actualizará `data/output/etapa9.zip`. Ejecuta `evaluar_siguiente()` otra vez para la siguiente pregunta.

También puedes utilizar `mostrar_pregunta(1)` y `guardar_pregunta(1, "2 0 1 0 0 2 1 0 0 1")` en vez de responder al cuadro de texto. Si pulsas Intro sin escribir nada, no se guarda una calificación inventada.

**Importante:** solo después de completar los 120 juicios se calculan Precision@5, Recall@5, nDCG@5 y MRR@10.


In [7]:
# Evaluación humana con entrada de texto estándar de Jupyter: no necesita ipywidgets.
from IPython.display import display, Markdown

def mostrar_pregunta(numero=1):
    numero=int(numero)
    if not (1 <= numero <= len(consultas)):
        raise ValueError(f'El número de pregunta debe estar entre 1 y {len(consultas)}')
    q=consultas[numero-1]
    datos=pd.read_csv(JUICIOS,dtype=str).fillna('')
    subset=datos.loc[datos.pregunta==q].set_index('mcid')
    print('\n'+'='*75)
    print(f'PREGUNTA {numero}/{len(consultas)}: {q}')
    print('='*75)
    for pos,mcid in enumerate(orden_ciego(q),start=1):
        r=subset.loc[mcid]
        print(f'\n{pos:2d}. {str(r.titulo)}')
        if r.autor: print('    Autor:',r.autor)
        print('    Ficha:',r.ficha)
        if r.relevancia_0_1_2 in ('0','1','2'):
            print('    Calificación guardada:',r.relevancia_0_1_2)
    print('\nEscala: 0 = no relevante | 1 = relación indirecta | 2 = relevante directamente')
    print('Escribe 10 notas en orden, una por cada ficha, separadas por espacios.')
    print('Si no conoces una respuesta, no asignes 0 por defecto: cancela y revisa la ficha.')
    return q

def guardar_pregunta(numero, notas, evaluador=''):
    numero=int(numero)
    if not (1 <= numero <= len(consultas)):
        raise ValueError(f'Número fuera de rango 1-{len(consultas)}')
    if isinstance(notas,str):
        valores=[x for x in notas.replace(',',' ').replace(';',' ').split() if x]
    else:
        valores=[str(x).strip() for x in notas]
    if len(valores)!=len(ids) or any(x not in ('0','1','2') for x in valores):
        raise ValueError('Escribe exactamente 10 notas, cada una 0, 1 o 2. Ejemplo: 2 0 1 0 0 2 1 0 0 1')
    q=consultas[numero-1]
    datos=pd.read_csv(JUICIOS,dtype=str).fillna('')
    for mcid,nota in zip(orden_ciego(q),valores):
        mask=(datos.pregunta==q)&(datos.mcid==mcid)
        if mask.sum()!=1: raise ValueError(f'No se encuentra una fila única para {mcid}')
        datos.loc[mask,'relevancia_0_1_2']=nota
        if evaluador: datos.loc[mask,'evaluador']=evaluador
    datos.to_csv(JUICIOS,index=False,encoding='utf-8-sig')
    estado=actualizar_evaluacion()
    destino=crear_zip()
    print(f'✅ Pregunta {numero} guardada. Juicios: {estado["etiquetas_validas"]}/120. Archivo: {destino}')
    if estado['etiquetas_pendientes']==0:
        print('\n¡Evaluación completa! Resultados:')
        display(pd.read_csv(SALIDA/'metricas_resumen.csv'))
    return estado

def evaluar_siguiente():
    datos=pd.read_csv(JUICIOS,dtype=str).fillna('')
    for numero,q in enumerate(consultas,start=1):
        sub=datos[datos.pregunta==q]
        if not sub.relevancia_0_1_2.isin(['0','1','2']).all():
            mostrar_pregunta(numero)
            try:
                notas=input(f'Notas para la pregunta {numero} (10 números 0/1/2; Intro cancela): ').strip()
            except (KeyboardInterrupt,EOFError):
                print('Sin cambios; puedes retomar cuando quieras.')
                return None
            if not notas:
                print('Sin cambios. Ejecuta evaluar_siguiente() para continuar.')
                return None
            return guardar_pregunta(numero,notas)
    print('✅ Las 120 etiquetas están completas.')
    estado=actualizar_evaluacion()
    print('ZIP actualizado:',crear_zip())
    if (SALIDA/'metricas_resumen.csv').exists():
        display(pd.read_csv(SALIDA/'metricas_resumen.csv'))
    return estado

print('Evaluación compatible con Jupyter: sin widgets.')
print('Escribe evaluar_siguiente() en una celda nueva para comenzar.')
print('Puedes revisar la primera pregunta con mostrar_pregunta(1).')


Evaluación compatible con Jupyter: sin widgets.
Escribe evaluar_siguiente() en una celda nueva para comenzar.
Puedes revisar la primera pregunta con mostrar_pregunta(1).


In [8]:
evaluar_siguiente()


PREGUNTA 1/12: ¿Qué obras catalogadas tratan sobre el aymara?

 1. Historia natural y moral de las Indias : en que se tratan las cosas notables del cielo y elementos, metales, plantas, y animales dellas y los ritos, y ceremonias, leyes y govierno, y guerras de los indios
    Autor: Acosta, José de
    Ficha: https://www.memoriachilena.gob.cl/602/w3-article-8652.html

 2. Con mi humilde devoción : bailes chinos en Chile central
    Autor: Mercado, Claudio | Rondón, Víctor
    Ficha: https://www.memoriachilena.gob.cl/602/w3-article-9893.html

 3. El término Ylla y su potencial simbólico en el Tawantinsuyu : una reflexión acerca de la presencia inca en Caspana (río Loa, Desierto de Atacama)
    Autor: Manríquez Soto, Viviana
    Ficha: https://www.memoriachilena.gob.cl/602/w3-article-81260.html

 4. Extirpación de la idolatria del Piru
    Autor: Arriaga, Pablo José de
    Ficha: https://www.memoriachilena.gob.cl/602/w3-article-8655.html

 5. Parte primera de la Chrónica del Perú : que t